In [2]:
df = catalog.load('data')

[12/23/23 14:51:05] INFO     Loading data from 'data' (CSVDataSet)...                           ]8;id=271815;file:///Users/air/Desktop/BetaStar/env/lib/python3.10/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=110388;file:///Users/air/Desktop/BetaStar/env/lib/python3.10/site-packages/kedro/io/data_catalog.py#345\345]8;;\

In [5]:
df.columns


Index(['node_id', 'target', 'participation', 'in_mod_deg', 'beta_star',
       'edge_contr', 'cada', 'kl', 'l1', 'l2', 'hd', 'kl2', 'l12', 'l22',
       'hd2', 'lcc', 'bc', 'dc', 'ndc', 'cc', 'ec', 'eccen', 'core'],
      dtype='object')

In [7]:
X = df.drop(['node_id', 'target'], axis=1)
y = df['target']
y = y.map(lambda x: int(x))

In [11]:
y


0       1
1       0
2       0
3       0
4       0
       ..
9995    1
9996    0
9997    0
9998    1
9999    0
Name: target, Length: 10000, dtype: int64

In [15]:
import torch
from torch.nn import Linear
import torch.nn.functional as F

In [12]:
d_in = len(X.columns)
d_h = d_in -5
d_o = 2

In [16]:
def accuracy(y_pred, y_true):
    return torch.sum(y_pred==y_true)/len(y_true)

class MLP(torch.nn.Module):

    def __init__(self,dim_in, dim_h, dim_out):
        super().__init__()
        self.linear1 = Linear(dim_in, dim_h)
        self.linear2 = Linear(dim_h, dim_out)
    
    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return F.log_softmax(x, dim=1)

    def fit(self, data, epochs):
        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=0.01, weight_decay=5e-4)
        self.train()
        for epoch in range(epochs+1):
            optimizer.zero_grad()
            out = self(data.x)
            loss = criterion(out[data.train_mask], data.y[data.train_mask])
            acc = accuracy(out[data.train_mask].argmax(dim=1), data.y[data.train_mask])
            loss.backward()
            optimizer.step()
            if epoch % 20 == 0:
                val_loss = criterion(out[data.val_mask], data.y[data.val_mask])
                val_acc = accuracy(out[data.val_mask].argmax(dim=1),data.y[data.val_mask])
                print(f'Epoch {epoch:>3} | Train Loss: {loss:.3f}  | Train Acc: {acc*100:>5.2f}% | Val Loss:{val_loss:.2f} | Val Acc: {val_acc*100:.2f}%')
        
    def test(self,data):
        self.eval()
        out = self(data.x)
        return accuracy(out.argmax(dim=1)[data.test_mask],data.y[data.test_mask])
    
    

In [19]:
model = MLP(d_in,d_h,d_o)

In [23]:
tensor = torch.tensor(X.values)

In [24]:
tensor


tensor([[ 7.2612e-02,  5.1158e+00,  7.1551e-02,  ...,  1.9969e-01,
          4.0000e+00,  1.2000e+01],
        [ 5.5461e-01,  1.4986e+01,  6.4898e-01,  ...,  1.9950e-01,
          4.0000e+00,  1.2000e+01],
        [ 5.3456e-01,  1.9957e+01,  5.9021e-01,  ...,  1.0852e-01,
          4.0000e+00,  1.2000e+01],
        ...,
        [ 4.4000e-01, -3.2710e-01,  5.3856e-01,  ...,  2.6391e-03,
          5.0000e+00,  5.0000e+00],
        [ 2.8000e-01, -5.0932e-01,  3.6992e-01,  ...,  1.4115e-03,
          5.0000e+00,  5.0000e+00],
        [ 6.8000e-01, -2.4679e-01,  7.6671e-01,  ...,  2.5474e-03,
          5.0000e+00,  5.0000e+00]], dtype=torch.float64)